# Maximum altitude

## Initialization

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
import ray
import numba as nb
import numpy as np
import xarray.ufuncs as xrf
import matplotlib.pyplot as plt
import pickle

from tqdm.auto import trange
from math import radians, pi, sin
from numba.experimental import jitclass
from IPython.display import display, JSON
import ipywidgets as widgets

from ray.rllib.agents import ppo
from ray.rllib.models import ModelCatalog
from ray.rllib.models.tf.tf_modelv2 import TFModelV2
from ray.rllib.models.tf.fcnet import FullyConnectedNetwork
from ray.tune.registry import register_env

from cw.filters import smooth_signal
from cw.vdom import hyr
from traj1.logger import Logger
from environment import LauncherV1SubOrbital, Stage, AP_PITCH_RATE_CONTROL

Instructions for updating:
non-resource variables are not supported in the long term


## Configuration

In [3]:
hyr(ray.init(dashboard_host="0.0.0.0"))

2021-05-13 22:55:00,580	INFO services.py:1267 -- View the Ray dashboard at http://0.0.0.0:8265


dict:

node_ip_address: str:

10.10.3.2

raylet_ip_address: str:

10.10.3.2

redis_address: str:

10.10.3.2:6379

object_store_address: str:

/tmp/ray/session_2021-05-13_22-54-59_041296_3120/sockets/plasma_store

raylet_socket_name: str:

/tmp/ray/session_2021-05-13_22-54-59_041296_3120/sockets/raylet

webui_url: str:

0.0.0.0:8265

session_dir: str:

/tmp/ray/session_2021-05-13_22-54-59_041296_3120

metrics_export_port: int:

64739

node_id: str:

b96da6265416edb00a0dc36e6d96e7b6035a52999b71182fbb584fda

In [1]:
ray.shutdown()

NameError: name 'ray' is not defined

In [5]:
register_env("LauncherV1SubOrbital", LauncherV1SubOrbital)

In [6]:
class CustomModel(TFModelV2):
    def __init__(self, obs_space, action_space, num_outputs, model_config, name):
        super().__init__(obs_space, action_space, num_outputs, model_config, name)

        self.fcnet = FullyConnectedNetwork(
            self.obs_space,
            self.action_space,
            num_outputs,
            model_config,
            name="fcnet")

    def forward(self, input_dict, state, seq_lens):
        return self.fcnet(input_dict, state, seq_lens)
    
    def value_function(self):
        return self.fcnet.value_function()

ModelCatalog.register_custom_model("custom_model", CustomModel)

In [ ]:
trainer = ppo.PPOTrainer(env="LauncherV1SubOrbital", config={
    "num_workers": 4,
    "num_gpus": 1,
    "batch_mode": "complete_episodes",
    "env_config": {},
    "model": {
        "custom_model": "custom_model",
        "custom_model_config": {},
    },
})

In [8]:
# trainer.load_checkpoint("./results/checkpoint_000371/checkpoint-371")

## RL Training

In [8]:
training_fig = plt.figure(figsize=(8, 2))
reward_ax = plt.gca()
plt.tight_layout()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [9]:
training_history = {
    "episode_reward_max": [],
    "episode_reward_min": [],
    "episode_reward_mean": [],
    "episode_len_mean": []
}

In [11]:
with open("./2.pickle", "wb") as f:
    pickle.dump(training_history, f)

In [10]:
out = widgets.Output()
out_checkpoint = widgets.Output()
display(out)
display(out_checkpoint)

for i in trange(1000 * 4):
    result = trainer.train()

    if i % 10 == 0:
        checkpoint = trainer.save("./results")
        out_checkpoint.clear_output(True)
        with out_checkpoint:
            display(f"checkpoint saved at {checkpoint}")

    out.clear_output(True)
    with out:
        display(hyr(result))

    for k in training_history.keys():
        training_history[k].append(result[k])
        
    # Plot reward history
    reward_ax.clear()
    reward_ax.plot(training_history["episode_reward_mean"])
    reward_ax.plot(training_history["episode_reward_max"])
    reward_ax.plot(training_history["episode_reward_min"])
    reward_ax.set_xlim((0, len(training_history["episode_reward_mean"])-1))
    training_fig.canvas.draw()


Output()

Output()

  0%|          | 0/4000 [00:00<?, ?it/s]

2021-05-13 22:55:52,445	WARNING deprecation.py:33 -- DeprecationWarning: `SampleBatch.data[..]` has been deprecated. Use `SampleBatch[..]` instead. This will raise an error in the future!
<ipython-input-10-211f29a2dcf4>:27: UserWarning: Attempting to set identical left == right == 0 results in singular transformations; automatically expanding.
  reward_ax.set_xlim((0, len(training_history["episode_reward_mean"])-1))


## Greedy evaluation

In [12]:
env = Environment(environment_configuration)

logger = Logger()
logger.register_time_attribute(env.sim, "t")
logger.register(env.sim, "env", [
    "h",
    "gamma_e", "theta_e", "theta_i_dot",
    "ap_comm_gamma_e", "ap_comm_theta_e",
    "action_autopilot_mode", "action_autopilot_reference", "vii", "xii", "fii_thrust", "mass", "mass_dot"])

In [13]:
episode_reward = 0
done = False
obs = env.reset()
while not done:
    action = trainer.compute_action(obs)
    obs, reward, done, info = env.step(action)
    episode_reward += reward
    logger.step()

episode_result = logger.episode_finish()
print(episode_reward)
episode_result

0.7042675419775867


<xarray.Dataset>
Dimensions:                         (d_2_0: 2, t: 464)
Coordinates:
  * t                               (t) float64 0.05 0.1 0.15 ... 23.15 23.2
Dimensions without coordinates: d_2_0
Data variables: (12/13)
    env_h                           (t) float64 0.0 -0.004062 ... 1.025e+03
    env_gamma_e                     (t) float64 -1.571 0.5631 ... 1.609 1.605
    env_theta_e                     (t) float64 1.069 1.119 ... 2.653 2.949
    env_theta_i_dot                 (t) float64 1.0 -1.0 -1.0 ... -1.0 1.0 1.0
    env_ap_comm_gamma_e             (t) float64 nan nan nan nan ... nan nan nan
    env_ap_comm_theta_e             (t) float64 nan nan nan nan ... nan nan nan
    ...                              ...
    env_action_autopilot_reference  (t) float64 1.0 -1.0 -1.0 ... -1.0 1.0 1.0
    env_vii                         (t, d_2_0) float64 -4.975e-18 ... 84.79
    env_xii                         (t, d_2_0) float64 1.064e-10 ... 1.738e+06
    env_fii_thrust                  (t, d_2_0) float64 3.268 5.963 ... 0.0 0.0
    env_mass                        (t) float64 1.2 1.2 1.199 ... 1.0 0.9998 1.0
    env_mass_dot                    (t) float64 -0.008665 -0.008665 ... 0.0 0.0

In [14]:
plt.figure()
smooth_signal(episode_result.env_action_autopilot_reference, wn=0.8).plot.line(x="t")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [18]:
plt.figure()
episode_result.env_theta_e.plot.line(x="t")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [16]:
plt.figure()
episode_result.env_h.plot.line(x="t")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [17]:
plt.figure()
plt.plot(result["hist_stats"]['episode_reward'])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [9]:
hyr(result)

dict:

episode_reward_max: float:

12717.627862528512

episode_reward_min: float:

724.773520008237

episode_reward_mean: numpy.float64:

5408.14551827484

episode_len_mean: numpy.float64:

12892.43

episode_media: dict:

episodes_this_iter: int:

4

policy_reward_min: dict:

policy_reward_max: dict:

policy_reward_mean: dict:

custom_metrics: dict:

hist_stats: dict:

episode_reward: list:

[0] float:

7505.203794956368

[1] float:

11072.00475198938

[2] float:

8612.14382989487

[3] float:

5572.336620102922

[4] float:

4843.752136511191

[5] float:

12225.875337529054

[6] float:

724.773520008237

[7] float:

1726.606503080471

[8] float:

5379.701772666273

[9] float:

4967.612564860715

[10] float:

2851.0445692856974

[11] float:

2507.613759204599

[12] float:

3696.718935510553

[13] float:

4293.643397670654

[14] float:

10406.325715787096

[15] float:

3777.389151379627

[16] float:

9442.710431465059

[17] float:

899.3596770961074

[18] float:

3054.963549572203

[19] float:

1591.0146461889578

[20] float:

1676.2065203637117

[21] float:

2991.7908437860724

[22] float:

1940.6051164697285

[23] float:

3487.809308594663

[24] float:

1343.2972955616278

[25] float:

2633.5837400510422

[26] float:

5043.462652882659

[27] float:

1894.1357772728852

[28] float:

2718.35622332636

[29] float:

2991.4375932595326

[30] float:

4764.7734846265275

[31] float:

2024.7611151754438

[32] float:

2786.78013713548

[33] float:

9995.161271222036

[34] float:

6083.172774342031

[35] float:

3383.1840579186796

[36] float:

2672.6606473679553

[37] float:

2923.3878319145992

[38] float:

6206.825653917136

[39] float:

2229.7644868703915

[40] float:

3337.722787370209

[41] float:

6109.369219878318

[42] float:

5028.764813260978

[43] float:

2907.838836126496

[44] float:

3480.7374596265895

[45] float:

5185.3712926233975

[46] float:

5848.422685121913

[47] float:

4077.9661239052884

[48] float:

3758.183157945705

[49] float:

5507.301998746972

[50] float:

6159.629710499982

[51] float:

3112.478891742678

[52] float:

4532.165499870675

[53] float:

5576.834792957624

[54] float:

8045.014717531962

[55] float:

2315.882863946295

[56] float:

4855.340163824772

[57] float:

4949.286885961849

[58] float:

6368.124087521892

[59] float:

4120.669553229655

[60] float:

6759.191026656256

[61] float:

7606.881232184404

[62] float:

9869.848038791732

[63] float:

3178.616845415265

[64] float:

5565.81698195845

[65] float:

6224.99016743641

[66] float:

8789.34430234133

[67] float:

6531.862306276161

[68] float:

7065.674157482094

[69] float:

5758.657870772666

[70] float:

8439.68027265033

[71] float:

6095.14704657571

[72] float:

8136.958910731709

[73] float:

8395.289890614313

[74] float:

10765.20016078294

[75] float:

6561.446406901661

[76] float:

5801.758534376439

[77] float:

5556.721055209698

[78] float:

9008.533993983323

[79] float:

4112.744629158933

[80] float:

8215.427301207239

[81] float:

7054.806038164325

[82] float:

7824.489214242628

[83] float:

6674.3327700917325

[84] float:

5878.587945741878

[85] float:

4363.882187051438

[86] float:

8591.585306006222

[87] float:

4704.482620512639

[88] float:

4107.451332000147

[89] float:

6992.586877291179

[90] float:

12717.627862528512

[91] float:

5286.449737829961

[92] float:

5050.960350497778

[93] float:

5773.291634032072

[94] float:

6729.239055511557

[95] float:

4443.0583787166825

[96] float:

4759.82243716144

[97] float:

5917.614952455133

[98] float:

10319.339867492337

[99] float:

4966.095390137415

episode_lengths: list:

[0] int:

15002

[1] int:

17462

[2] int:

17320

[3] int:

12343

[4] int:

14912

[5] int:

19394

[6] int:

2423

[7] int:

3861

[8] int:

14059

[9] int:

13710

[10] int:

15097

[11] int:

5835

[12] int:

7689

[13] int:

9979

[14] int:

16954

[15] int:

15722

[16] int:

17182

[17] int:

5285

[18] int:

10950

[19] int:

6112

[20] int:

6442

[21] int:

8624

